# import Python packages

In [ ]:
!pip install faiss-cpu

import huggingface_hub, datasets, transformers, os, sys, json, random, re, nltk, time, requests, faiss

import pandas as pd
import numpy as np

from transformers import pipeline
from sentence_transformers import SentenceTransformer

from nltk.tokenize import sent_tokenize

from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from dotenv import load_dotenv

nltk.download('punkt_tab')

In [ ]:
import llm_functions, rag_functions, evaluation_metrics, data_functions
# FYI: It's necessary to have these .PY files in the same directory as this Python Notebook and an __init__.py file needs to be in the same directory to import the functions from these Python files.
# Email Sara Kingsley or the Teaching Assistants with any issues importing these.

from llm_functions import LLM_requesters
from rag_functions import ragLLM
from  evaluation_metrics import RagMetrics
from data_functions import DataFunctions

llm = LLM_requesters()
ragllm = ragLLM()
metrics = RagMetrics()
datafunc = DataFunctions()


In [ ]:
# load in the required API keys from your .env file:
load_dotenv()

openai = os.getenv('OPENAI_API_KEY')       # OpenAI API Key.
HF_TOKEN = os.getenv('HF_TOKEN')    # HuggingFaceHub Authentication Token.
HF_INF_TOKEN = os.getenv("HF_INF_TOKEN")   # HuggingFace Inference Endpoint Token.
OPENAI_API_KEY = openai
gemma_endpoint = os.getenv("gemma_endpoint_url")

In [ ]:
model_01 = "gpt-3.5-turbo"
model_02 = "gpt-4o"
model_03="meta-llama/Llama-3.2-1B"

In [ ]:
input_message = {
	"inputs": "Can you please let us know more details about your ",
	"parameters": {
		"max_new_tokens": 100}
}

output = llm.query_llm_model(input_message, HF_INF_TOKEN, model_endpoint=gemma_endpoint)

print(output)

# Coding Tasks Required to Answer Bonus Questions

In [ ]:
# import the Resume dataset into the Python environment:

df_path = "Resume.csv"   # Note you will need to change this path to the location of the Resume.csv file on your local machine

df=pd.read_csv(df_path)

In [ ]:
# change the column name to 'text'; this will format the dataset in the fashion required by some of the functions in this notebook
df.rename(columns = {'Resume_str':'text'}, inplace = True)

In [ ]:
''' Create a sample dataset from the Resume dataset:
Since the Resume dataset contains many rows, we will create a sample dataset with a smaller number of rows to work with.
'''

# Function to select 10 random rows from each group, where group is defined by the column named 'Category', and the row selection is from the column named 'text
def select_random_rows(group, n=10):
    return group.sample(n=min(n, len(group)))

# Applying the function to each group in the column named 'Category':
sample_resume_df = df.groupby('Category').apply(lambda x: select_random_rows(x['text'], 10)).reset_index(drop=False)

sample_resume_df.tail(100)

In [ ]:
# # Split resumes into sentences
sentences_df = datafunc.split_resumes_to_sentences(sample_resume_df, 'text')

In [ ]:
# create a list to store the chunks of text that we will create next:
resume_sentences = []

# split the text in each row of the 'text' column into sentences and store the sentences in the list:
for row in sample_resume_df['text']:
    sentences = datafunc.split_text_into_sentences(row)
    resume_sentences.extend(sentences)

In [ ]:
# define the model for creating embeddings:
model = SentenceTransformer('bert-base-nli-mean-tokens')

# create embeddings for the sentences:
sentence_embeddings = model.encode(sentences_df['sentence'])
sentence_embeddings.shape

In [ ]:
len(set(sentences))

### Create a FAISS Index, Add the sentence embeddings to the index and create a Vector store:

In [ ]:

d = sentence_embeddings.shape[1]

nb = len(set(sentences))

nq = 10000
np.random.seed(1234)             # make reproducible
xb = np.random.random((nb, d)).astype('float32')
nlist = 100

import faiss


index = faiss.IndexFlatL2(d)
index

index.add(sentence_embeddings)

index.ntotal

index.train(sentence_embeddings)

index.is_trained  # check if index is now trained


## Define the models we will use in the experiments:

In [ ]:
model_01 = "gpt-3.5-turbo"
model_02 = "gpt-4o"
model_03="meta-llama/Llama-3.2-1B"

In [ ]:
sentence_embeddings

## Define the Queries for the Experiments:

In [ ]:
d = sentence_embeddings.shape[1]
nb = len(set(sentences))
nq = 10000
np.random.seed(1234)             # make reproducible
xb = np.random.random((nb, d)).astype('float32')
nlist = 100
index = faiss.IndexFlatL2(d)
index
index.add(sentence_embeddings)
index.train(sentence_embeddings)
print(index.is_trained)

def get_sys_message(user_prompt, k):
    model = SentenceTransformer('bert-base-nli-mean-tokens')
    k=k
    xq = model.encode([user_prompt])
    D, I = index.search(xq, k)  # search
    first_index = I[0]  # Get the first index from I
    rag_search_results = sentences_df['sentence'].iloc[first_index].sum()
    return rag_search_results

# Create 3 system prompts to use in a call to different LLMs by querying the vector store with the queries given to you in the Bonus Task Instructions:
(the questions below are examples; don't use these to complete the bonus tasks)

In [ ]:
sys_messageA = get_sys_message(user_prompt="Which resume has the most software skills listed?", k=5)
sys_messageA

In [ ]:
sys_messageB = get_sys_message(user_prompt="Which resume has the most accounting skills listed?", k=5)
sys_messageB

In [ ]:
sys_messageC = get_sys_message(user_prompt="Which resume has the most biology skills listed?", k=5)
sys_messageC

# Request responses from 4 different LLMs, following the Bonus Task Instructions:
(the questions below are examples; don't use these to complete the bonus tasks)

In [ ]:

textA=ragllm.rag_llm_openai(model=model_01, user_prompt="Which resume has the most software skills listed?", k=10, task="summarize", system_message=sys_message)

In [ ]:
textA

In [ ]:

textB= ragllm.rag_llm_openai(model=model_02, user_prompt="Which resume has the most software skills listed?", k=10, task="summarize", system_message=sys_message)

In [ ]:
textB

In [ ]:

textC=ragllm.rag_llm(model=model_03, user_prompt="Which resume has the most software skills listed?", k=10, task="summarize", system_message=sys_message)

In [ ]:
textD = "<insert call to LLM of your chocie>"

## Evaluate the Information Retrieved and LLM responses by comparing the similarity of the responses across models

In [ ]:
def compare_text_similarity(text_a, text_b, text_c):
    """
    Compares the similarity between three texts using TF-IDF vectors and cosine similarity.

    Parameters:
    - text_a (str): Text A
    - text_b (str): Text B
    - text_c (str): Text C

    Returns:
    - A dictionary with similarity scores between Text A & Text B, A & C, and B & C.
    """
    # Initialize the vectorizer and transform texts into TF-IDF vectors
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text_a, text_b, text_c])

    # Calculate cosine similarity
    similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

    # Similarity between Text A and B, A and C, and then B and C
    similarity_scores = {
        "A_B": similarity_matrix[0, 1],
        "A_C": similarity_matrix[0, 2],
        "B_C": similarity_matrix[1, 2]
    }

    return similarity_scores

def compare_text_similarity_response2context(om, model_response):
    """
    Compares the similarity between three texts using TF-IDF vectors and cosine similarity.

    Parameters:
    - text_a (str): Text A
    - text_b (str): Text B
    - text_c (str): Text C

    Returns:
    - A dictionary with similarity scores between Text A & Text B, A & C, and B & C.
    """
    # Initialize the vectorizer and transform texts into TF-IDF vectors
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([om, model_response])

    # Calculate cosine similarity
    similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

    # Similarity between Text A and B, A and C, and then B and C
    similarity_scores = {
        "A_B": similarity_matrix[0, 1],
        #"A_C": similarity_matrix[0, 2],
        #"B_C": similarity_matrix[1, 2]
    }

    return similarity_scores

## Compare the Similarity of LLM Response Messages A, B, and C:

In [ ]:
similarities = compare_text_similarity(text_a=textA, text_b=textB, text_c=textC)

print("LLM Response Similarity Scores:")
for pair, score in similarities.items():
    print(f"{pair}: {score:.4f}")

## Compare the Similarity of System Messages A, B, and C:

In [ ]:
similaritiesOM =compare_text_similarity(text_a=sys_messageA, text_b=sys_messageB, text_c=sys_messageC)

print("System Prompt Similarity -to-  System Prompt Similarity Scores:")
for pair, score in similaritiesOM.items():
    print(f"{pair}: {score:.4f}")

## Compare the Similarity of System Messages A, B, and C to LLM Response Messages A, B and C:

In [ ]:
similaritiesOM_LLM_A =compare_text_similarity_response2context(om=sys_messageA, model_response=textA)
similaritiesOM_LLM_B =compare_text_similarity_response2context(om=sys_messageB, model_response=textB)
similaritiesOM_LLM_C =compare_text_similarity_response2context(om=sys_messageC, model_response=textC)

print("System Prompt to LLM Response \n Similarity Scores \n Model 1:")
for pair, score in similaritiesOM_LLM_A.items():
    print(f"{pair}: {score:.4f}")
print("\n")
print("System Prompt to LLM Response\n Similarity Scores \n Model 2:")
for pair, score in similaritiesOM_LLM_B.items():
    print(f"{pair}: {score:.4f}")
print("\n")
print("System Prompt to LLM Response\n Similarity Scores \n Model 3:")
for pair, score in similaritiesOM_LLM_C.items():
    print(f"{pair}: {score:.4f}")

# **UPDATED CODE FOR BONUS QUESTIONS**

In [ ]:
!pip install faiss-cpu

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import faiss
import os

from sentence_transformers import SentenceTransformer
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Add this line to download the missing resource

# Load API keys
load_dotenv()
OPENAI_API_KEY = openai
HF_TOKEN = HF_TOKEN
openai_key = OPENAI_API_KEY # Directly use the defined variable
your_huggingface_token = HF_TOKEN # Directly use the defined variable

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the Resume dataset
df = pd.read_csv("path/to/Resume.csv")

# Rename column for consistency
df.rename(columns={'Resume_str': 'text'}, inplace=True)

print(f"📊 Total resumes: {len(df)}")
print(f"📋 Categories: {df['Category'].unique()}")

# Function to sample resumes
def select_random_rows(group, n=10):
    return group.sample(n=min(n, len(group)), random_state=42)

# Create sample dataset (10 resumes per category)
sample_resume_df = df.groupby('Category').apply(
    lambda x: select_random_rows(x[['text', 'Category']], 10)
).reset_index(drop=True)

print(f"\n✅ Sample dataset created: {len(sample_resume_df)} resumes")
print(f"📂 Categories in sample: {sample_resume_df['Category'].value_counts()}")

In [ ]:
# Function to split text into sentences
def split_text_into_sentences(text):
    """Split text into sentences using NLTK"""
    sentences = sent_tokenize(text)
    return sentences

# Function to create sentences dataframe with identifiers
def split_resumes_to_sentences(df, text_column):
    """
    Split resumes into sentences with unique identifiers
    """
    sentences_list = []

    for idx, row in df.iterrows():
        sentences = split_text_into_sentences(row[text_column])

        for sentence in sentences:
            sentences_list.append({
                'resume_id': idx,
                'category': row['Category'] if 'Category' in row else None,
                'sentence': sentence
            })

    sentences_df = pd.DataFrame(sentences_list)
    return sentences_df

# Split resumes into sentences
print("📝 Chunking resumes into sentences...")
sentences_df = split_resumes_to_sentences(sample_resume_df, 'text')

print(f"✅ Created {len(sentences_df)} sentence chunks")
print(f"📄 Average sentences per resume: {len(sentences_df) / len(sample_resume_df):.1f}")

In [ ]:
# # Load embedding model
# print(" Loading embedding model...")
# model = SentenceTransformer('bert-base-nli-mean-tokens', token=HF_TOKEN)

# # Create embeddings for all sentences
# print("🔢 Creating embeddings (this may take a minute)...")
# sentence_embeddings = model.encode(
#     sentences_df['sentence'].tolist(),
#     show_progress_bar=True
# )

# print(f"✅ Embeddings created!")
# print(f"📐 Shape: {sentence_embeddings.shape}")
# print(f"   - Sentences: {sentence_embeddings.shape[0]}")
# print(f"   - Dimensions: {sentence_embeddings.shape[1]}")

In [ ]:
# ============================================
# UPDATED EMBEDDING CODE WITH FASTER MODEL
# ============================================

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load the FASTER, BETTER model
print("🤖 Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2', token=HF_TOKEN)

# Create embeddings
print(f"🔢 Creating embeddings for {len(sentences_df)} sentences...")
print("⏱️  Estimated time: 2-5 minutes")

sentence_embeddings = model.encode(
    sentences_df['sentence'].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"✅ Embeddings created!")
print(f"📐 Shape: {sentence_embeddings.shape}")

# Build FAISS index with correct dimension (384)
d = sentence_embeddings.shape[1]
print(f"\n🏗️  Building FAISS vector store (dimension: {d})...")

index = faiss.IndexFlatL2(d)
index.add(sentence_embeddings.astype('float32'))
index.train(sentence_embeddings.astype('float32'))

print(f"✅ Vector store ready!")
print(f"📊 Vectors stored: {index.ntotal}")

In [ ]:
def get_rag_context(query, k=10):
    """
    Query the vector store and get relevant context

    Parameters:
    - query: Question to ask
    - k: Number of results to retrieve

    Returns:
    - context: Combined text from top k results
    """
    # Create embedding for query
    query_embedding = model.encode([query])

    # Search vector store
    D, I = index.search(query_embedding.astype('float32'), k)

    # Get the matching sentences
    retrieved_sentences = sentences_df.iloc[I[0]]['sentence'].tolist()

    # Combine into context
    context = " ".join(retrieved_sentences)

    return context


# Test the function
test_query = "What programming languages are mentioned?"
test_context = get_rag_context(test_query, k=5)
print(f"🔍 Test query: {test_query}")
print(f"📄 Retrieved context length: {len(test_context)} characters")
print(f"Preview: {test_context[:200]}...")

In [ ]:
# THE 3 REQUIRED BONUS QUERIES
bonus_queries = [
    "What computer programming languages are listed on IT technology resumes?",
    "What computer programming languages are listed on nurse resumes?",
    "What computer programming languages are listed on teacher resumes?"
]

print("📋 Bonus Queries:")
for i, query in enumerate(bonus_queries, 1):
    print(f"  {i}. {query}")

# Get RAG context for each query
print("\n🔍 Retrieving context from vector store...")
contexts = {}
for i, query in enumerate(bonus_queries, 1):
    context = get_rag_context(query, k=20)  # Get more context
    contexts[f"Query_{i}"] = {
        'query': query,
        'context': context
    }
    print(f"  ✓ Query {i}: Retrieved {len(context)} characters")

In [ ]:
# ===== USING OPENAI MODELS =====

from openai import OpenAI

client = OpenAI(api_key=openai)

def query_openai_with_rag(model_name, query, context, task="list and summarize"):
    """
    Query OpenAI model with RAG context
    """
    system_prompt = f"""You are a helpful assistant analyzing resumes.
    Use the following context from resumes to answer the question.

    Context: {context}
    """

    user_prompt = f"""{task} the following: {query}

    Provide a concise, specific answer based only on the context provided."""

    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=300,
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"


# ===== USING HUGGINGFACE MODELS =====

from transformers import pipeline

def query_huggingface_with_rag(model_name, query, context):
    """
    Query HuggingFace model with RAG context
    """
    try:
        # Determine pipeline type based on model name
        if 't5' in model_name.lower() or 'flan' in model_name.lower():
            pipeline_task = 'text2text-generation'
        else:
            pipeline_task = 'text-generation'

        # Load model with token and appropriate pipeline task
        generator = pipeline(pipeline_task, model=model_name, token=your_huggingface_token)

        prompt = f"""Context: {context[:1000]}

Question: {query}

Answer based on the context:"""

        # For text2text-generation models, the input format can be simpler
        if pipeline_task == 'text2text-generation':
            # For T5, the input is often just the question, or context + question
            response = generator(prompt, max_new_tokens=200)[0]['generated_text']
        else: # text-generation
            response = generator(prompt, max_new_tokens=200)[0]['generated_text']

        # Extract just the answer part
        if "Answer based on the context:" in response:
            answer = response.split("Answer based on the context:")[1].strip()
        else:
            answer = response

        return answer
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Define the 4 LLMs we'll use
llm_models = {
    'LLM1_GPT3.5': 'gpt-3.5-turbo',
    'LLM2_GPT4': 'gpt-4o-mini',  # Using mini version (cheaper)
    'LLM3_Llama': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', # Changed to an accessible Llama model
    'LLM4_Flan': 'distilgpt2' # Keeping this working HuggingFace model for now
}

# Store all responses
all_responses = {}

print("🤖 Querying LLMs with RAG context...")
print("=" * 80)

for query_key, query_info in contexts.items():
    query = query_info['query']
    context = query_info['context']

    print(f"\n{query_key}: {query}")
    print("-" * 80)

    responses = {}

    # Query OpenAI models (LLM1 and LLM2)
    for llm_name in ['LLM1_GPT3.5', 'LLM2_GPT4']:
        model_name = llm_models[llm_name]
        print(f"  Querying {llm_name} ({model_name})...")

        response = query_openai_with_rag(model_name, query, context)
        responses[llm_name] = response

        print(f"    Response: {response[:150]}...")

    # Query HuggingFace models (LLM3 and LLM4)
    for llm_name in ['LLM3_Llama', 'LLM4_Flan']:
        model_name = llm_models[llm_name]
        print(f"  Querying {llm_name} ({model_name})...")

        # No special handling needed for Gemini anymore, directly call HF function
        response = query_huggingface_with_rag(model_name, query, context)
        responses[llm_name] = response

        print(f"    Response: {response[:150]}...")

    all_responses[query_key] = {
        'query': query,
        'context': context,
        'responses': responses
    }

print("\n✅ All LLM queries completed!")

In [ ]:
# from transformers import pipeline, set_seed
# import torch

# # Set seed for reproducibility
# set_seed(42)

# # Define the 4 LLMs - GUARANTEED TO WORK
# llm_models = {
#     'LLM1_GPT3.5': 'gpt-3.5-turbo',
#     'LLM2_GPT4': 'gpt-4o-mini',
#     'LLM3_GPT2': 'gpt2',
#     'LLM4_FlanT5': 'google/flan-t5-base'
# }

# # Store all responses
# all_responses = {}

# print("🤖 Querying LLMs with RAG context...")
# print("=" * 80)

# for query_key, query_info in contexts.items():
#     query = query_info['query']
#     context = query_info['context']

#     print(f"\n{query_key}: {query}")
#     print("-" * 80)

#     responses = {}

#     # ==========================================
#     # OPENAI MODELS (LLM1 and LLM2)
#     # ==========================================
#     for llm_name in ['LLM1_GPT3.5', 'LLM2_GPT4']:
#         model_name = llm_models[llm_name]
#         print(f"  Querying {llm_name} ({model_name})...")

#         response = query_openai_with_rag(model_name, query, context)
#         responses[llm_name] = response

#         print(f"    ✓ Response: {response[:150]}...")

#     # ==========================================
#     # HUGGINGFACE MODELS (LLM3 and LLM4)
#     # ==========================================
#     for llm_name in ['LLM3_GPT2', 'LLM4_FlanT5']:
#         model_name = llm_models[llm_name]
#         print(f"  Querying {llm_name} ({model_name})...")

#         response = query_huggingface_with_rag(model_name, query, context)
#         responses[llm_name] = response

#         print(f"    ✓ Response: {response[:150]}...")

#     all_responses[query_key] = {
#         'query': query,
#         'context': context,
#         'responses': responses
#     }

# print("\n✅ All LLM queries completed!")

In [ ]:
# Define evaluation functions
def compare_text_similarity(text_a, text_b, text_c):
    """Compare similarity between three texts using TF-IDF and cosine similarity"""
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text_a, text_b, text_c])

    similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

    return {
        "A_B": similarity_matrix[0, 1],
        "A_C": similarity_matrix[0, 2],
        "B_C": similarity_matrix[1, 2]
    }

def compare_response_to_context(context, response):
    """Compare LLM response similarity to original context"""
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([context, response])

    similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

    return similarity_matrix[0, 1]


# Evaluate for each query
evaluation_results = {}

print("\n📊 EVALUATION METRICS")
print("=" * 80)

for query_key, data in all_responses.items():
    print(f"\n{query_key}: {data['query'][:50]}...")
    print("-" * 80)

    responses = data['responses']
    context = data['context']

    # Compare LLM responses to each other (for first 2 LLMs)
    llm1_response = responses['LLM1_GPT3.5']
    llm2_response = responses['LLM2_GPT4']
    llm3_response = responses['LLM3_Llama']

    response_similarity = compare_text_similarity(
        llm1_response,
        llm2_response,
        llm3_response
    )

    print("\n  📈 LLM Response Similarity:")
    print(f"    LLM1 ↔ LLM2: {response_similarity['A_B']:.4f}")
    print(f"    LLM1 ↔ LLM3: {response_similarity['A_C']:.4f}")
    print(f"    LLM2 ↔ LLM3: {response_similarity['B_C']:.4f}")

    # Compare each response to context
    print("\n  📈 Response-to-Context Similarity:")
    for llm_name, response in responses.items():
        context_sim = compare_response_to_context(context, response)
        print(f"    {llm_name}: {context_sim:.4f}")

    evaluation_results[query_key] = {
        'response_similarity': response_similarity,
        'context_similarity': {
            llm: compare_response_to_context(context, resp)
            for llm, resp in responses.items()
        }
    }

print("\n✅ Evaluation complete!")

In [ ]:
# Create comprehensive results table
results_summary = []

for query_key, data in all_responses.items():
    for llm_name, response in data['responses'].items():
        results_summary.append({
            'Query': query_key,
            'LLM': llm_name,
            'Response': response[:200] + "...",
            'Response_Length': len(response),
            'Context_Similarity': evaluation_results[query_key]['context_similarity'][llm_name]
        })

results_df = pd.DataFrame(results_summary)

# Save to CSV
results_df.to_csv('bonus_results.csv', index=False)

print("💾 Results saved to 'bonus_results.csv'")
print("\n📊 Results Preview:")
print(results_df[['Query', 'LLM', 'Context_Similarity']])